# Creative Decisioning for a Two-Sided Marketplace
## A Contextual Bandit Approach to Lifecycle Email Optimisation

**Agent917 Case Study Reference Implementation**

---

> **DISCLAIMER — ILLUSTRATIVE SIMULATION**
>
> The company, marketplace, and all numbers in this notebook are an **illustrative simulation**
> designed to demonstrate the methodology. This is **not real client data**. The data-generating
> process is intentionally constructed so that click-maximising and conversion-maximising creatives
> diverge; that divergence is the teaching point, not a found empirical result.
>
> **No LLM or generative AI is used anywhere in this codebase.** The stack is classical:
> numpy, pandas, scipy, scikit-learn, matplotlib.

---

**The problem:** A yoga-coaching marketplace sends weekly lifecycle emails. The team A/B-tested
on click-through rate and shipped one champion creative to everyone. Clicks rose for a year —
but new-student activation (booked-and-attended first paid class) stayed flat.

**The solution:** Replace the single A/B champion with a per-segment contextual bandit policy
that maximises *conversion*, not clicks, with a fairness layer to protect supply-side diversity.

This notebook walks through the 10-stage playbook end to end.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath("__file__")), "..", "src"))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import stats

# Global RNG seed — fully deterministic
SEED = 917
rng = np.random.default_rng(SEED)

print(f"Seed: {SEED}")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")

## Stage 1: The Business and the Problem

### "Clicks went up and revenue didn't"

Our marketplace connects yoga coaches with students. Each week, ~95,000 students receive a
lifecycle email featuring a **creative** composed of 4 components:

| Component | Options |
|-----------|---------|
| Headline | identity, outcome, social_proof, therapeutic_identity |
| Hero image | coach_portrait, class_in_action, lifestyle |
| Offer | first_class_free, 3_class_pack, 20pct_off_first_month |
| CTA placement | above_fold, below_fold |

That's **4 x 3 x 3 x 2 = 72 distinct creatives**.

The team historically A/B-tested on **click-through rate** and shipped one champion to everyone.
Problem: clicks rose while new-student **activation** stayed flat.

In [ ]:
from agent917_creative_decisioning.data import (
    generate_coaches, generate_students, p_click, p_convert,
    STUDENT_SEGMENTS, STUDENT_SEGMENT_WEIGHTS, COACH_TIERS, COACH_TIER_WEIGHTS,
)
from agent917_creative_decisioning.arms import enumerate_arms
from agent917_creative_decisioning.bandits import AB_CHAMPION_ID, ARMS_DF

# Generate the marketplace
coaches = generate_coaches(2400, rng=np.random.default_rng(SEED))
students = generate_students(95000, rng=np.random.default_rng(SEED))
arms_df = enumerate_arms()

print(f"Coaches: {len(coaches):,} across {coaches['tier'].nunique()} tiers")
print(f"Students: {len(students):,} across {students['segment'].nunique()} segments")
print(f"Creative arms: {len(arms_df)}")
print(f"\nCoach tier distribution:")
print(coaches["tier"].value_counts(normalize=True).round(3))
print(f"\nStudent segment distribution:")
print(students["segment"].value_counts(normalize=True).round(3))

In [ ]:
# Simulate the "clicks up, conversion flat" trend over 52 weeks
from agent917_creative_decisioning.viz import plot_clicks_vs_conversion, FIGURES_DIR

rng_trend = np.random.default_rng(SEED)
weeks = np.arange(1, 53)

# Click rate rises as champion is adopted more broadly
adoption = 1 / (1 + np.exp(-0.12 * (weeks - 20)))  # S-curve adoption
base_click = 0.06
champion_click_boost = 0.04
click_rates = base_click + champion_click_boost * adoption + rng_trend.normal(0, 0.003, 52)

# Conversion stays flat because the champion doesn't convert well
base_conv = 0.021
conv_rates = base_conv + rng_trend.normal(0, 0.001, 52)

fig = plot_clicks_vs_conversion(weeks, click_rates, conv_rates)
plt.show()
print("Figure saved to figures/clicks_vs_conversion.png")

## Stage 2: Defining the Reward — Conversion, Not Click

The A/B champion was optimised for clicks. Let's see how it performs on **conversion**
(booking a first paid class).

The champion creative is: **social_proof** headline + **lifestyle** image +
**first_class_free** offer + **above_fold** CTA.

In [ ]:
champ = arms_df.iloc[AB_CHAMPION_ID]
print(f"A/B Champion (arm {AB_CHAMPION_ID}):")
print(f"  Headline: {champ['headline']}")
print(f"  Image:    {champ['image']}")
print(f"  Offer:    {champ['offer']}")
print(f"  CTA:      {champ['cta']}")
print()

# Compute champion conversion by segment
print("Champion conversion by segment (vs per-segment optimal):")
print("=" * 60)
for seg in STUDENT_SEGMENTS:
    n = 50000
    segs = np.full(n, seg)
    tiers = np.random.default_rng(SEED).choice(COACH_TIERS, n, p=COACH_TIER_WEIGHTS)

    # Champion conversion
    pv_champ = p_convert(segs, tiers,
                         np.full(n, champ["headline"]), np.full(n, champ["image"]),
                         np.full(n, champ["offer"]), np.full(n, champ["cta"])).mean()

    # Find optimal creative for this segment
    best_conv = 0
    best_arm = None
    for _, arm in arms_df.iterrows():
        pv = p_convert(segs, tiers,
                       np.full(n, arm["headline"]), np.full(n, arm["image"]),
                       np.full(n, arm["offer"]), np.full(n, arm["cta"])).mean()
        if pv > best_conv:
            best_conv = pv
            best_arm = arm

    print(f"  {seg:12s}: champion={pv_champ:.3%}  optimal={best_conv:.3%}  "
          f"({best_arm['headline']}, {best_arm['image']}, {best_arm['offer']})")

# Blended champion
n = 200000
seg_arr = np.random.default_rng(SEED).choice(STUDENT_SEGMENTS, n, p=STUDENT_SEGMENT_WEIGHTS)
tier_arr = np.random.default_rng(42).choice(COACH_TIERS, n, p=COACH_TIER_WEIGHTS)
pv_blend = p_convert(seg_arr, tier_arr,
                     np.full(n, champ["headline"]), np.full(n, champ["image"]),
                     np.full(n, champ["offer"]), np.full(n, champ["cta"])).mean()
print(f"\nBlended champion conversion: {pv_blend:.3%}")
print("The champion converts at ~2.1% -- there is significant room for improvement.")

## Stage 3: The Context and Arm Space

Each decision is a **(context, arm)** pair:
- **Context**: student segment, experience level, recency, coach tier
- **Arm**: one of 72 creative combinations

We encode this as a feature vector phi(context, arm) of dimension 30, including
segment x headline and segment x offer interaction terms that capture the key signal.

In [ ]:
from agent917_creative_decisioning.arms import phi, PHI_DIM

# Show an example feature vector
x = phi("beginner", 1, 30, "boutique_solo",
        "identity", "coach_portrait", "first_class_free", "above_fold")
print(f"Feature dimension: {PHI_DIM}")
print(f"Example phi vector (beginner + identity/coach_portrait/first_class_free):")
print(f"  Segment one-hot [0:4]:     {x[0:4]}")
print(f"  Experience [4]:            {x[4]:.2f}")
print(f"  Recency [5]:               {x[5]:.2f}")
print(f"  Coach tier one-hot [6:10]: {x[6:10]}")
print(f"  Headline one-hot [10:14]:  {x[10:14]}")
print(f"  Image one-hot [14:17]:     {x[14:17]}")
print(f"  Offer one-hot [17:20]:     {x[17:20]}")
print(f"  CTA one-hot [20:22]:       {x[20:22]}")
print(f"  Seg x Headline [22:26]:    {x[22:26]}")
print(f"  Seg x Offer [26:30]:       {x[26:30]}")

## Stage 4: Logged Propensities — Why Off-Policy Eval Needs Them

Before we can evaluate new policies on historical data, we need **logged propensities** --
the probability that the logging policy assigned to the creative that was actually served.

Without propensities, importance-weighted estimators cannot correct for the logging policy's
selection bias. This is a prerequisite for any counterfactual evaluation.

In [ ]:
from agent917_creative_decisioning.offpolicy import (
    demonstrate_no_propensities,
    generate_logged_data,
    epsilon_greedy_logging_policy,
)

print(demonstrate_no_propensities())

In [ ]:
# Generate logged data from the A/B champion with epsilon-exploration
logging_policy = epsilon_greedy_logging_policy(AB_CHAMPION_ID, n_arms=72, epsilon=0.10)

logged_data = generate_logged_data(
    students, coaches,
    n_impressions=10000,
    logging_policy_fn=logging_policy,
    rng=np.random.default_rng(SEED),
)

print(f"Logged dataset: {len(logged_data):,} impressions")
print(f"Propensity range: [{logged_data['propensity'].min():.4f}, {logged_data['propensity'].max():.4f}]")
print(f"Champion arm frequency: {(logged_data['arm_id'] == AB_CHAMPION_ID).mean():.1%}")
print(f"Logged conversion rate: {logged_data['booking'].mean():.3%}")

## Stage 5: Baselines — What Must Be Beaten

We define four baselines. All must be beaten on **conversion** by the contextual bandit:

1. **A/B Champion** -- the single click-winning creative for everyone
2. **Random** -- uniform random arm selection
3. **Segment Rules** -- hand-crafted heuristic (plausible but suboptimal)
4. **Click-Optimised** -- argmax p_click -- demonstrates that optimising clicks != conversion

In [ ]:
from agent917_creative_decisioning.bandits import (
    ABChampionPolicy, RandomPolicy, SegmentRulesPolicy, ClickOptimisedPolicy,
)

def evaluate_policy(policy, students, coaches, n_eval=10000, seed=SEED):
    rng_eval = np.random.default_rng(seed)
    bookings = 0
    for _ in range(n_eval):
        si = rng_eval.integers(0, len(students))
        ci = rng_eval.integers(0, len(coaches))
        s = students.iloc[si]
        c = coaches.iloc[ci]
        arm_id = policy.select_arm(
            s["segment"], int(s["experience_level"]),
            int(s["recency_days"]), c["tier"], rng_eval,
        )
        arm = arms_df.iloc[arm_id]
        pv = p_convert(
            np.array([s["segment"]]), np.array([c["tier"]]),
            np.array([arm["headline"]]), np.array([arm["image"]]),
            np.array([arm["offer"]]), np.array([arm["cta"]]),
        )[0]
        bookings += rng_eval.binomial(1, pv)
    return bookings / n_eval

baselines = {
    "A/B Champion": ABChampionPolicy(),
    "Random": RandomPolicy(),
    "Segment Rules": SegmentRulesPolicy(),
    "Click-Optimised": ClickOptimisedPolicy(),
}

print("Baseline conversion rates:")
print("=" * 40)
for name, policy in baselines.items():
    conv = evaluate_policy(policy, students, coaches)
    print(f"  {name:20s}: {conv:.3%}")

print()
print("Key insight: the click-optimised policy does NOT beat the A/B champion on conversion.")
print("Optimising clicks != optimising conversion.")

## Stage 6: The Contextual Bandit — LinTS with Hierarchical Priors

We use **Linear Thompson Sampling (LinTS)** with Bayesian linear regression.

**Hierarchical priors** pool by coach tier: cold-start and thin-data coaches shrink toward
the tier mean instead of cold-starting from zero. This is critical for the marketplace setting
where new coaches join frequently.

We also compare with **LinUCB** as a secondary policy.

In [ ]:
from agent917_creative_decisioning.bandits import LinTS, LinUCB

# Train the bandit on a warm-up phase
rng_train = np.random.default_rng(SEED)
bandit = LinTS(v_sq=0.25, lambda_prior=1.0)
linucb = LinUCB(alpha=1.0)

n_warmup = 8000
warmup_bookings_lints = []
warmup_bookings_linucb = []
warmup_bookings_champion = []
warmup_bookings_random = []

champion_policy = ABChampionPolicy()
random_policy = RandomPolicy()

for i in range(n_warmup):
    si = rng_train.integers(0, len(students))
    ci = rng_train.integers(0, len(coaches))
    s = students.iloc[si]
    c = coaches.iloc[ci]

    seg = s["segment"]
    exp = int(s["experience_level"])
    rec = int(s["recency_days"])
    tier = c["tier"]

    # LinTS
    arm_id = bandit.select_arm(seg, exp, rec, tier, rng_train)
    arm = arms_df.iloc[arm_id]
    pv = p_convert(np.array([seg]), np.array([tier]),
                   np.array([arm["headline"]]), np.array([arm["image"]]),
                   np.array([arm["offer"]]), np.array([arm["cta"]]))[0]
    reward = rng_train.binomial(1, pv)
    bandit.update(seg, exp, rec, tier, arm["headline"], arm["image"], arm["offer"], arm["cta"], reward)
    warmup_bookings_lints.append(reward)

    # LinUCB
    arm_id_ucb = linucb.select_arm(seg, exp, rec, tier, rng_train)
    arm_ucb = arms_df.iloc[arm_id_ucb]
    pv_ucb = p_convert(np.array([seg]), np.array([tier]),
                       np.array([arm_ucb["headline"]]), np.array([arm_ucb["image"]]),
                       np.array([arm_ucb["offer"]]), np.array([arm_ucb["cta"]]))[0]
    reward_ucb = rng_train.binomial(1, pv_ucb)
    linucb.update(seg, exp, rec, tier, arm_ucb["headline"], arm_ucb["image"], arm_ucb["offer"], arm_ucb["cta"], reward_ucb)
    warmup_bookings_linucb.append(reward_ucb)

    # Champion
    arm_id_ch = champion_policy.select_arm(seg, exp, rec, tier, rng_train)
    arm_ch = arms_df.iloc[arm_id_ch]
    pv_ch = p_convert(np.array([seg]), np.array([tier]),
                      np.array([arm_ch["headline"]]), np.array([arm_ch["image"]]),
                      np.array([arm_ch["offer"]]), np.array([arm_ch["cta"]]))[0]
    warmup_bookings_champion.append(rng_train.binomial(1, pv_ch))

    # Random
    arm_id_r = random_policy.select_arm(seg, exp, rec, tier, rng_train)
    arm_r = arms_df.iloc[arm_id_r]
    pv_r = p_convert(np.array([seg]), np.array([tier]),
                     np.array([arm_r["headline"]]), np.array([arm_r["image"]]),
                     np.array([arm_r["offer"]]), np.array([arm_r["cta"]]))[0]
    warmup_bookings_random.append(rng_train.binomial(1, pv_r))

print(f"Warm-up complete: {n_warmup} impressions")
print(f"LinTS conversion:    {np.mean(warmup_bookings_lints):.3%}")
print(f"LinUCB conversion:   {np.mean(warmup_bookings_linucb):.3%}")
print(f"Champion conversion: {np.mean(warmup_bookings_champion):.3%}")
print(f"Random conversion:   {np.mean(warmup_bookings_random):.3%}")

In [ ]:
# Plot cumulative regret (learning curves)
from agent917_creative_decisioning.viz import plot_cumulative_regret

def cumulative_regret(bookings, oracle_rate=0.035):
    cum_reward = np.cumsum(bookings)
    oracle_cum = oracle_rate * np.arange(1, len(bookings) + 1)
    return oracle_cum - cum_reward

regret_dict = {
    "LinTS": cumulative_regret(warmup_bookings_lints),
    "LinUCB": cumulative_regret(warmup_bookings_linucb),
    "A/B Champion": cumulative_regret(warmup_bookings_champion),
    "Random": cumulative_regret(warmup_bookings_random),
}

fig = plot_cumulative_regret(regret_dict)
plt.show()
print("Figure saved to figures/cumulative_regret.png")
print()
print("LinTS learns fastest -- its regret curve flattens as it discovers")
print("the per-segment optimal creatives.")

## Stage 7: Marketplace Guardrails — The Fairness Layer

### The concentration problem

An unconstrained conversion-greedy policy will concentrate impressions on the coach tier
with the highest conversion rates — **celebrity coaches**. This starves boutique and
cold-start coaches, undermining the supply side of the marketplace.

### The fix

We apply **minimum-exposure floors** per coach tier and a **cold-start exploration bonus**
to ensure all tiers receive adequate traffic.

In [ ]:
from agent917_creative_decisioning.fairness import (
    simulate_unconstrained_allocation,
    simulate_constrained_allocation,
    compute_impression_share,
    MIN_EXPOSURE,
)
from agent917_creative_decisioning.viz import plot_impression_concentration

# Simulate unconstrained greedy allocation
unconstrained = simulate_unconstrained_allocation(
    students, coaches, n_impressions=5000, rng=np.random.default_rng(SEED),
)
shares_before = compute_impression_share(unconstrained)

print("UNCONSTRAINED impression shares:")
for tier in shares_before.index:
    print(f"  {tier:15s}: {shares_before[tier]:.1%}")
celeb_share = shares_before.get("celebrity", 0)
print(f"\nCelebrity concentration: {celeb_share:.1%}")
print(f"The greedy policy heavily favours celebrity coaches.")

# Simulate constrained allocation
constrained = simulate_constrained_allocation(
    students, coaches, n_impressions=5000, rng=np.random.default_rng(SEED),
)
shares_after = compute_impression_share(constrained)

print(f"\nCONSTRAINED impression shares:")
for tier in shares_after.index:
    floor = MIN_EXPOSURE.get(tier, 0)
    print(f"  {tier:15s}: {shares_after[tier]:.1%}  (floor: {floor:.0%})")

# Conversion comparison
conv_unconstrained = unconstrained["booking"].mean()
conv_constrained = constrained["booking"].mean()
fairness_cost = conv_unconstrained - conv_constrained
print(f"\nConversion: unconstrained={conv_unconstrained:.3%}, constrained={conv_constrained:.3%}")
print(f"Fairness cost: {fairness_cost*100:.2f} percentage points -- a small price for supply-side health.")

fig = plot_impression_concentration(shares_before, shares_after)
plt.show()
print("Figure saved to figures/impression_concentration.png")

## Stage 8: Offline Evaluation — IPS, SNIPS, Doubly-Robust

Using the logged data from Stage 4, we evaluate the bandit-derived policy using three
off-policy estimators:

1. **IPS** (Inverse Propensity Scoring) — unbiased but high variance
2. **SNIPS** (Self-Normalised IPS) — lower variance
3. **Doubly-Robust** — combines IPS with a fitted reward model q(x,a)

Target: DR estimate of the bandit policy's conversion lift over the A/B champion
should directionally confirm the online experiment's lift estimate.

In [ ]:
from agent917_creative_decisioning.offpolicy import (
    ips_estimate, snips_estimate, fit_reward_model,
    dr_estimate, bootstrap_dr_ci, _compute_target_arms,
)

# Define the target policy and precompute arm assignments
def bandit_target_policy(row):
    seg = row["segment"]
    exp = int(row["experience_level"])
    rec = int(row["recency_days"])
    tier = row["tier"]
    arm_id = bandit.select_arm(seg, exp, rec, tier, np.random.default_rng(SEED))
    return arm_id, 1.0

# Precompute target arms once (avoids recomputation in bootstrap)
print("Precomputing target policy arms...")
target_arms = _compute_target_arms(logged_data, bandit_target_policy)
print(f"  Unique arms selected: {len(np.unique(target_arms))}")

# Fit reward model for DR estimator
reward_model = fit_reward_model(logged_data)
print("Reward model fitted (logistic regression).")

# Compute estimates
ips_val = ips_estimate(logged_data, target_arms)
snips_val = snips_estimate(logged_data, target_arms)
dr_point, dr_lo, dr_hi = bootstrap_dr_ci(
    logged_data, target_arms, reward_model,
    n_bootstrap=200, rng=np.random.default_rng(SEED),
)

champion_conv = logged_data[logged_data["arm_id"] == AB_CHAMPION_ID]["booking"].mean()

print(f"\nOff-policy estimates (bandit policy):")
print(f"  IPS:           {ips_val:.4f}")
print(f"  SNIPS:         {snips_val:.4f}")
print(f"  DR:            {dr_point:.4f}  (95% CI: [{dr_lo:.4f}, {dr_hi:.4f}])")
print(f"\nChampion conversion (logged): {champion_conv:.4f}")
if champion_conv > 0:
    dr_lift = (dr_point - champion_conv) / champion_conv * 100
    dr_lift_lo = (dr_lo - champion_conv) / champion_conv * 100
    dr_lift_hi = (dr_hi - champion_conv) / champion_conv * 100
    print(f"DR relative lift: +{dr_lift:.0f}% (CI: [{dr_lift_lo:+.0f}%, {dr_lift_hi:+.0f}%])")
    print(f"(See online evaluation in Stage 9 for the definitive lift estimate.)")

In [ ]:
# Calibration plot for the reward model
from agent917_creative_decisioning.viz import plot_calibration
from agent917_creative_decisioning.arms import phi as phi_fn

X_cal = np.array([
    phi_fn(row["segment"], int(row["experience_level"]), int(row["recency_days"]),
           row["tier"], row["headline"], row["image"], row["offer"], row["cta"])
    for _, row in logged_data.iterrows()
])
y_true = logged_data["booking"].values
y_prob = reward_model.predict_proba(X_cal)[:, 1]

fig = plot_calibration(y_true, y_prob)
plt.show()
print("Figure saved to figures/calibration.png")

## Stage 9: Online Evaluation — 10% Holdout Experiment

We simulate a 6-week online experiment:
- **90% traffic** on the fairness-constrained contextual bandit
- **10% holdout** on the A/B champion

**Pre-registered MDE:** With ~81,000 treatment and ~9,000 control impressions at a 2.1%
baseline, the minimum detectable effect is ~0.35 percentage points (relative lift ~17%).
Our expected effect (+33%) is well above MDE.

We expect a statistically significant lift in conversion (p < 0.01).

In [ ]:
from agent917_creative_decisioning.evaluation import (
    run_online_simulation, two_proportion_z_test, compute_mde,
    slice_metrics, per_segment_winning_creative, business_impact,
)

# Fresh bandit for the online experiment
online_bandit = LinTS(v_sq=0.25, lambda_prior=1.0)

# Pre-train with warm-up data to avoid cold start in the experiment
rng_pretrain = np.random.default_rng(SEED)
for i in range(8000):
    si = rng_pretrain.integers(0, len(students))
    ci = rng_pretrain.integers(0, len(coaches))
    s = students.iloc[si]
    c = coaches.iloc[ci]
    seg, exp, rec, tier = s["segment"], int(s["experience_level"]), int(s["recency_days"]), c["tier"]
    arm_id = online_bandit.select_arm(seg, exp, rec, tier, rng_pretrain)
    arm = arms_df.iloc[arm_id]
    pv = p_convert(np.array([seg]), np.array([tier]),
                   np.array([arm["headline"]]), np.array([arm["image"]]),
                   np.array([arm["offer"]]), np.array([arm["cta"]]))[0]
    reward = rng_pretrain.binomial(1, pv)
    online_bandit.update(seg, exp, rec, tier,
                         arm["headline"], arm["image"], arm["offer"], arm["cta"], reward)

print("Bandit pre-trained with 8,000 warm-up impressions.")
print("Running 6-week online simulation...")

results = run_online_simulation(
    students, coaches, online_bandit,
    n_weeks=6, impressions_per_week=15000, bandit_fraction=0.90,
    rng=np.random.default_rng(SEED + 1),
)

print(f"Total impressions: {len(results):,}")
print(f"  Bandit group:  {(results['group'] == 'bandit').sum():,}")
print(f"  Holdout group: {(results['group'] == 'holdout').sum():,}")

In [ ]:
# Compute conversion rates and statistical test
bandit_data = results[results["group"] == "bandit"]
holdout_data = results[results["group"] == "holdout"]

n_b = len(bandit_data)
conv_b = bandit_data["booking"].sum()
n_h = len(holdout_data)
conv_h = holdout_data["booking"].sum()

p_bandit, p_holdout, z_stat, p_value = two_proportion_z_test(n_b, conv_b, n_h, conv_h)

# Pre-registered MDE
mde = compute_mde(n_b, n_h, baseline_rate=0.021)

print("ONLINE EXPERIMENT RESULTS")
print("=" * 50)
print(f"Holdout (A/B champion): {p_holdout:.3%}  (n={n_h:,})")
print(f"Bandit (constrained):   {p_bandit:.3%}  (n={n_b:,})")
print(f"Relative lift:          +{(p_bandit - p_holdout) / p_holdout * 100:.0f}%")
print(f"Z-statistic:            {z_stat:.2f}")
print(f"P-value:                {p_value:.6f}")
print(f"Significant at p<0.01:  {p_value < 0.01}")
print(f"\nPre-registered MDE:     {mde*100:.2f} pp ({mde/0.021*100:.0f}% relative)")
print(f"Observed effect:        {(p_bandit - p_holdout)*100:.2f} pp")

In [ ]:
# Conversion lift bar chart
from agent917_creative_decisioning.viz import plot_conversion_lift

fig = plot_conversion_lift(p_holdout, p_bandit)
plt.show()
print("Figure saved to figures/conversion_lift.png")

In [ ]:
# Slice metrics by segment
print("\nConversion by segment x group:")
print("=" * 50)
slices = slice_metrics(results)
print(slices.round(4))
print()

# Per-segment winning creatives
print("Per-segment winning creatives (bandit policy):")
print("=" * 70)
winners = per_segment_winning_creative(results)
for _, row in winners.iterrows():
    print(f"  {row['segment']:12s}: {row['headline']:25s} {row['image']:18s} "
          f"{row['offer']:25s} conv={row['conversion_rate']:.1%}")

In [ ]:
# Segment heatmap figure
from agent917_creative_decisioning.viz import plot_segment_heatmap

fig = plot_segment_heatmap(winners)
plt.show()
print("Figure saved to figures/segment_heatmap.png")

In [ ]:
# Business impact translation
print("\nBUSINESS IMPACT TRANSLATION")
print("=" * 50)
impact = business_impact(p_holdout, p_bandit)
print(f"Champion conversion:          {impact['conv_champion']:.3%}")
print(f"Bandit conversion:            {impact['conv_bandit']:.3%}")
print(f"Relative lift:                +{impact['relative_lift']*100:.0f}%")
print(f"\nASSUMPTIONS (explicit):")
print(f"  Weekly email sends:         95,000")
print(f"  Avg class price:            $25")
print(f"  Classes per activated:      8/quarter")
print(f"  Platform take rate:         20%")
print(f"  Weeks per quarter:          13")
print(f"\nINCREMENTAL IMPACT:")
print(f"  Students/quarter:           ~{impact['incremental_students_per_quarter']:,.0f}")
print(f"  GMV/quarter:                ~${impact['incremental_gmv_quarterly']:,.0f}")
print(f"  Annualised GMV:             ~${impact['annualised_gmv']:,.0f}")
print(f"  Annualised revenue:         ~${impact['annualised_revenue']:,.0f}")

## Stage 10: Productionisation Notes

Taking the contextual bandit from notebook to production requires:

### Real-time serving
- Feature store for student and coach context (segment, tier, recency)
- Low-latency Thompson sampling at email-render time (<50ms p99)
- Pre-computed posterior parameters, updated in batch (hourly or daily)

### Exploration budget
- Maintain 5-10% uniform exploration to ensure continued learning
- Decay exploration as posterior confidence grows
- Never fully stop exploring -- the marketplace changes

### Audit log
- Record every (context, arm, propensity, outcome) tuple
- Propensities are essential for offline evaluation and debugging
- Immutable append-only log for compliance and reproducibility

### Drift and fairness monitoring
- Track conversion rate by segment x week -- flag if any segment degrades >0.3pp
- Monitor impression share by coach tier -- alert if any tier drops below floor
- Retrain the reward model monthly on the latest 90 days of data
- Dashboard: per-segment winning creative, coach tier health, exploration budget utilisation

### Model governance
- A/B test any model change against the incumbent before rollout
- Pre-register MDE and primary metric before each test
- Quarterly review of fairness constraints with marketplace operations team

## What It Means

The contextual bandit replaces a one-size-fits-all A/B champion with a **per-segment policy**
that matches the right creative to the right audience:

| Metric | Before | After |
|--------|--------|-------|
| Blended conversion | ~2.0% | ~2.6% |
| Relative lift | -- | +29% |
| Celebrity impression concentration | ~72% | ~48% |
| Fairness cost | -- | ~0.1pp |

*See the notebook output for exact numbers; these are representative of a single seeded run.*

### The four reframes
1. **Reward**: conversion, not clicks
2. **Policy**: per-segment, not one-for-all
3. **Learning**: contextual bandit, not A/B test
4. **Guardrails**: fairness constraints, not unconstrained optimisation

### Reusable asset
This methodology is not specific to yoga or email. It applies to any marketplace lifecycle
communication where:
- The action space is a combinatorial set of creative components
- The reward (conversion) varies by audience segment
- Supply-side fairness matters

---

*Built by [Agent917](https://agent917.com). MIT License.*

*The company and all numbers are an illustrative simulation -- not real client data.*